# Start here

The same five steps as the `pipelines` workbench, as ordinary notebooks — no
`@dsl.component`, no `Compiler().compile(...)`, no submit cell. Run them top to
bottom, change the parameters, break them.

| Notebook | Reads | Writes |
|---|---|---|
| `01_clean_data` | the three raw downloads | `clean/branches.csv`, `clean/population_demographics.csv`, `clean/census_demos.csv` |
| `02_fetch_isochrones` | `clean/branches.csv`, Feast | `features/isochrones.parquet`, Feast |
| `03_compute_coverage` | isochrones + tracts | `results/coverage_details.parquet` |
| `04_aggregate_scores` | `results/coverage_details.parquet` | `results/coverage_scores.parquet` |
| `05_train_model` | details + scores + branches + tracts | `models/coverage-model/model.json`, `results/coverage_predictions.*` |

Each step does the same work as its counterpart in the `pipelines` workbench
and produces the same objects in the same places. Two things are packaged
differently for Elyra; both are explained below and neither changes a result.

Every step hands off through S3, never through a variable in memory. That is
what makes them rearrangeable: a step does not care whether the one before it
ran five seconds ago in the next cell or last Tuesday in a pipeline.

## Building a pipeline

**1. Make a canvas.** File → New → Pipeline Editor, and save it as something
like `coverage.pipeline`. Elyra writes a JSON file next to your notebooks.

**2. Drag the notebooks on** in order and draw an arrow from each to the next.
The arrows are the only thing that sets execution order.

**3. Set the defaults once.** A node with no runtime image fails validation
with *"runtime image required"* — Elyra requires one on every node and does
not pick for you. Rather than setting it six times, click empty canvas space
to open **Pipeline Properties** (not node properties) and fill in **Generic
Node Defaults**. Every node inherits what you put there.

- **Runtime image** → `Runtime | Datascience | CPU | Python 3.12 | 2025.2`

  The exact strings come from your cluster, so check the dropdown rather than
  trusting this page. `Minimal` works too — every notebook installs its own
  dependencies in the first cell, so the image mostly just needs pip.

- **Kubernetes secrets** → all four, needed by every step:

  | Environment variable | Secret name | Secret key |
  |---|---|---|
  | `AWS_ACCESS_KEY_ID` | `cpl-s3` | `AWS_ACCESS_KEY_ID` |
  | `AWS_SECRET_ACCESS_KEY` | `cpl-s3` | `AWS_SECRET_ACCESS_KEY` |
  | `AWS_S3_ENDPOINT` | `cpl-s3` | `AWS_S3_ENDPOINT` |
  | `AWS_S3_BUCKET` | `cpl-s3` | `AWS_S3_BUCKET` |

**4. Override per node where it matters.** Right-click a node → Open
Properties. Only two steps need anything beyond the defaults:

- `02` — add `MAPBOX_TOKEN` from the `mapbox-token` secret, plus `MODE` and
  `TRAVEL_TIME` as environment variables.
- `05` — optionally `MODE`, `TRAVEL_TIME`, `NEW_BRANCH_LAT`, `NEW_BRANCH_LON`.
  All have defaults, so you can skip this on a first pass.

**5. Run it.** The play button submits to the Data Science Pipelines server and
the run appears under Pipelines in the dashboard like any other.

You do **not** need to configure MLflow on any node. See below.

## One notebook, three nodes

`02_fetch_isochrones` reads `MODE` and `TRAVEL_TIME` from the environment
rather than hardcoding them. Drop the same notebook on the canvas three times
and give each copy different **Environment Variables**:

| Node | `MODE` | `TRAVEL_TIME` |
|---|---|---|
| driving | `driving` | `5` |
| cycling | `cycling` | `10` |
| walking | `walking` | `10` |

Chain them in series, not side by side — Mapbox allows 300 requests a minute
and three parallel nodes will trip it. The KFP version runs its three pipelines
sequentially for the same reason.

Those three pairs are also exactly what `03` treats as live, via its
`ACTIVE_COMBOS` set. The isochrone cache is append-only and never forgets, so
if you add a fourth combination here, add it there too or it will be fetched
and then ignored.

`05_train_model` works the same way with `MODE`, `TRAVEL_TIME`,
`NEW_BRANCH_LAT` and `NEW_BRANCH_LON` — fan out several candidate locations and
compare them.

## MLflow

Nothing to configure per node. Two mechanisms cover the two ways a step runs:

- **Cells by hand in this workbench** — the Notebook carries
  `opendatahub.io/mlflow-instance`, which makes the controller inject
  `MLFLOW_TRACKING_URI`. `05` opens its own run under the
  `cpl-coverage-model` experiment.
- **As a pipeline node** — the pipeline server opens a parent run for the
  pipeline and a nested run for each task, and passes the task its
  `MLFLOW_RUN_ID`. `05` logs into *that* run instead of starting a competing
  one, so the metrics stay attached to the run that produced them.

The second depends on `spec.mlflow.injectUserEnvVars: true` on the DSPA. Left
at its default of `false`, the parent and nested runs still appear but task
containers get no credentials — so the runs show up empty and any
`mlflow.log_metric()` inside a step fails. It is set in the chart.

If neither applies, `05` skips logging and still writes the model to S3.

## Feast

`02` uses the Feast online store as its cache, the same as the KFP version.
One difference in how it gets there, and one trap worth knowing about.

**The config is written, not mounted.** The KFP component mounts
`feature_store.yaml` from the `feast-client-config` ConfigMap. An Elyra node
cannot mount a ConfigMap — node properties cover environment variables,
secrets, PVCs, annotations, labels and tolerations, and that is the entire
list. It does not need to: the file is twelve lines, so `02` writes it out
itself. The certificate it points at is the OpenShift service CA, already
projected into every pod at
`/var/run/secrets/kubernetes.io/serviceaccount/service-ca.crt` and byte for
byte identical to the one in the ConfigMap. The namespace is read from the
same directory, so nothing is pinned to one cluster.

**The client version has to match the server.** A plain `pip install feast`
resolves to an older client here, and an older client fails every call against
this feature server with a `422`. The failure is caught and logged rather than
raised — so it does not look like a bug. It looks like a cache that never hits,
and 80 Mapbox calls on every single run. `02` pins the client. To confirm what
the server is running:

```
oc get featurestore -n cpl-coverage -o jsonpath='{.items[0].status.feastVersion}'
```

## If something breaks

- **`KeyError: 'AWS_S3_ENDPOINT'`** — the node is missing its secret mapping,
  or the workbench started before the `cpl-s3` secret existed. Restart it.
- **A later step says a key does not exist** — the step before it has not run.
  Check the bucket, not the notebook.
- **`SystemExit: No coverage data to aggregate`** — `04` refusing to overwrite
  good results with an empty frame. Run `03`.
- **Mapbox 401** — `mapbox-token` is empty. It comes from
  `secrets.mapboxToken` in the chart's `values.yaml`.
- **Feast `422`, or "Feast lookup failed"** — client/server version mismatch.
  See above. The run still completes; it just refetches everything.
- **Nothing in the Pipelines UI after you hit run** — Elyra needs a runtime
  configuration. RHOAI creates one for workbenches in a project that has a
  pipeline server; if the dropdown is empty, the pipeline server is still
  starting.
- **An MLflow run appears but is empty** — the task could not authenticate.
  Check `spec.mlflow.injectUserEnvVars` on the DSPA.